In [ ]:
!pip install -r requirements.txt

In [ ]:
# ============================================================
# COLAB-COMPATIBLE LEAD GENERATION PIPELINE (FINAL FIX)
# ============================================================

import sys
import subprocess

# Install dependencies
print("📦 Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                      "openai>=1.12.0", "langgraph>=0.0.30", "pydantic>=2.5.0",
                      "pandas>=2.0.0", "openpyxl>=3.1.0", "requests>=2.31.0",
                      "beautifulsoup4>=4.12.0", "python-dotenv>=1.0.0",
                      "nest-asyncio"])
print("✅ Dependencies installed\n")

# Get API Key
import os
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("🔑 API Key loaded from Colab Secrets")
except:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        OPENAI_API_KEY = input("Enter OpenAI API Key: ").strip()

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError("❌ Invalid OpenAI API key")
print("✅ API Key validated\n")

# Import modules
print("🤖 Loading Lead Generation Agent...")
import pandas as pd
import json
import asyncio
import nest_asyncio

# Apply nest_asyncio
nest_asyncio.apply()

# Import the pipeline (lead_agent.py must be uploaded to Colab Files)
from lead_agent import LeadPipeline, AgentState
print("✅ Agent loaded\n")

# Check for uploaded CSV files
print("📂 Checking for uploaded CSV files...")
csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
if csv_files:
    print(f"✅ Found CSV files: {', '.join(csv_files)}")
    INPUT_SOURCE = csv_files[0]  # Use first CSV found
    print(f"📊 Using: {INPUT_SOURCE}\n")
else:
    # Create sample data if no CSV uploaded
    print("📝 No CSV found. Creating sample file...")
    sample_data = {
        'name': ['Sarah Chen', 'James Wilson', 'Maria Garcia', 'David Park', 'Emma Thompson'],
        'company': ['Stripe', 'Shopify', 'Notion', 'Figma', 'Databricks'],
        'title': ['VP of Product', 'Head of Engineering', 'Product Manager', 'Design Lead', 'Data Scientist'],
        'email': ['sarah.chen@stripe.com', 'james.w@shopify.com', 'maria@notion.so', 'david@figma.com', 'emma.t@databricks.com'],
        'website': ['https://stripe.com', 'https://shopify.com', 'https://notion.so', 'https://figma.com', 'https://databricks.com'],
        'industry': ['Financial Technology', 'E-commerce Platform', 'Productivity Software', 'Design Tools', 'Data Analytics'],
        'company_size': ['5000+', '10000+', '500-1000', '1000-5000', '5000+'],
        'location': ['San Francisco', 'Ottawa', 'San Francisco', 'San Francisco', 'San Francisco']
    }
    df = pd.DataFrame(sample_data)
    INPUT_SOURCE = 'sample_leads.csv'
    df.to_csv(INPUT_SOURCE, index=False)
    print(f"✅ Created: {INPUT_SOURCE}\n")

# Run pipeline
print("="*70)
print("🚀 STARTING LEAD GENERATION PIPELINE")
print("="*70)
print()

INPUT_TYPE = 'csv'

# Create pipeline and run
async def run_full_pipeline():
    pipeline = LeadPipeline(OPENAI_API_KEY)

    # Create initial state
    initial_state = AgentState(
        input_type=INPUT_TYPE,
        raw_input=INPUT_SOURCE
    )

    # Run the graph
    final_state = await pipeline.graph.ainvoke(initial_state)

    # Convert to output format
    output = []
    enriched_leads = final_state.get('enriched_leads', []) if isinstance(final_state, dict) else final_state.enriched_leads

    for lead in enriched_leads:
        lead_dict = lead.dict() if hasattr(lead, 'dict') else lead
        output.append({
            "lead_id": lead_dict["lead_id"],
            "name": lead_dict["name"],
            "company": lead_dict["company"],
            "title": lead_dict["title"],
            "email": lead_dict["email"],
            "website": lead_dict["website"],
            "industry": lead_dict["industry"],
            "company_size": lead_dict["company_size"],
            "location": lead_dict["location"],
            "company_summary": lead_dict["company_summary"],
            "score": round(lead_dict["score"], 1),
            "fit_reasons": lead_dict["fit_reasons"],
            "email_generated": lead_dict["email_generated"]
        })

    return output

# Run the pipeline
results = asyncio.run(run_full_pipeline())

print()
print("="*70)
print("✅ PIPELINE COMPLETE")
print("="*70)
print()

# Display results
print(f"📊 Processed {len(results)} leads\n")

if results:
    print("📋 SAMPLE ENRICHED LEAD:")
    print("="*70)
    print(json.dumps(results[0], indent=2))
    print()

print("\n📈 LEAD SUMMARY:")
print("="*70)
summary_df = pd.DataFrame([{
    'Name': r['name'],
    'Company': r['company'],
    'Score': r['score'],
    'Industry': r['industry'] or 'Unknown',
    'Email Subject': r['email_generated']['subject'][:50] + '...'
} for r in results])
print(summary_df.to_string(index=False))
print()

# Save outputs
print("💾 Saving outputs...")
with open('enriched_leads.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✅ Saved: enriched_leads.json")

output_df = pd.DataFrame(results)
output_df['email_subject'] = output_df['email_generated'].apply(lambda x: x['subject'])
output_df['email_body'] = output_df['email_generated'].apply(lambda x: x['body'])
output_df['fit_reasons_text'] = output_df['fit_reasons'].apply(lambda x: '; '.join(x))
output_df = output_df.drop(['email_generated', 'fit_reasons'], axis=1)
output_df.to_csv('enriched_leads.csv', index=False)
print("✅ Saved: enriched_leads.csv")

# Statistics
print()
print("📊 PIPELINE STATISTICS:")
print("="*70)
print(f"Total Leads Processed: {len(results)}")
print(f"Average Score: {sum(r['score'] for r in results) / len(results):.1f}/10")
print(f"High-Value Leads (>8.0): {len([r for r in results if r['score'] >= 8.0])}")
print(f"Emails Generated: {len([r for r in results if r['email_generated']])}")
print(f"Companies with Websites: {len([r for r in results if r['website']])}")
print()

print("🎉 Done! Check the output files for full results.")
print()
print("💡 TIP: Access results with:")
print("   - results[0] → First lead")
print("   - results[0]['email_generated'] → Generated email")
print("   - [r for r in results if r['score'] >= 8] → High-scoring leads")

📦 Installing dependencies...
✅ Dependencies installed

🔑 API Key loaded from Colab Secrets
✅ API Key validated

🤖 Loading Lead Generation Agent...
✅ Agent loaded

📂 Checking for uploaded CSV files...
✅ Found CSV files: sample_leads.csv
📊 Using: sample_leads.csv

🚀 STARTING LEAD GENERATION PIPELINE



/usr/lib/python3.12/html/parser.py:415: RuntimeWarning: coroutine 'main' was never awaited
  m = attrfind_tolerant.match(rawdata, k)



✅ PIPELINE COMPLETE

📊 Processed 5 leads

📋 SAMPLE ENRICHED LEAD:
{
  "lead_id": 1,
  "name": "Sarah Chen",
  "company": "Stripe",
  "title": "VP of Product",
  "email": "sarah.chen@stripe.com",
  "website": "https://stripe.com",
  "industry": "Financial Technology",
  "company_size": "5000+",
  "location": "San Francisco",
  "company_summary": "Stripe is a leading player in the financial technology (fintech) industry, specializing in providing payment processing solutions and financial infrastructure for businesses of all sizes. The company offers a suite of products and services that enable online and in-person payment acceptance, financial services integration, and the development of custom revenue models. With a strong market position, Stripe is widely adopted by millions of companies globally, positioning itself as a critical partner for businesses looking to enhance their revenue growth and streamline financial operations.",
  "score": 9.0,
  "fit_reasons": [
    "Sarah Chen is 

/tmp/ipython-input-3408010872.py:99: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  lead_dict = lead.dict() if hasattr(lead, 'dict') else lead


In [ ]:
!pip install -r requirements.txt

In [ ]:
# ============================================================
# CELL 2: LEAD GENERATION PIPELINE WITH ELITE AI PROMPTS
# Run Cell 1 first to install dependencies
# Works in: Google Colab AND Local Jupyter
# ============================================================

import os
import json
import asyncio
import nest_asyncio
from typing import List, Dict, Any, Optional, Union, Literal
from enum import Enum
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field, validator
from langgraph.graph import StateGraph, END
from openai import AsyncOpenAI
import re
import logging

# Apply nest_asyncio for Colab compatibility
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ============================================================
# STEP 1: GET API KEY (Auto-detects Colab vs Local)
# ============================================================

OPENAI_API_KEY = None
ENV_TYPE = None

try:
    # Try Google Colab first
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    ENV_TYPE = "Colab"
    print("🔑 API Key loaded from Colab Secrets")
except:
    # Try .env file (Local)
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    if OPENAI_API_KEY:
        ENV_TYPE = "Local"
        print("🔑 API Key loaded from .env file")
    else:
        # Manual input as fallback
        print("⚠️ No API key found in Colab Secrets or .env file")
        OPENAI_API_KEY = input("Enter OpenAI API Key: ").strip()
        ENV_TYPE = "Manual"

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError("❌ Invalid OpenAI API key. Please set OPENAI_API_KEY.")

print(f"✅ API Key validated (Environment: {ENV_TYPE})\n")

# ============================================================
# STEP 2: FILE UPLOAD INTERFACE
# ============================================================

print("📂 FILE UPLOAD")
print("="*70)

uploaded_file_path = None

if ENV_TYPE == "Colab":
    # Google Colab file upload
    print("🔵 Running in Google Colab")
    print("Please upload your file (CSV, Excel, or JSON):\n")

    from google.colab import files
    uploaded = files.upload()

    if uploaded:
        uploaded_file_path = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {uploaded_file_path}")
    else:
        raise ValueError("❌ No file uploaded. Please upload a CSV, Excel, or JSON file.")

else:
    # Local Jupyter - ask for file path
    print("🟢 Running in Local Jupyter")
    print("Enter the path to your file (CSV, Excel, or JSON):\n")

    uploaded_file_path = input("File path: ").strip()

    if not uploaded_file_path or not os.path.exists(uploaded_file_path):
        raise ValueError(f"❌ File not found: {uploaded_file_path}")

    print(f"✅ File found: {uploaded_file_path}")

print()

# Detect file type
file_extension = uploaded_file_path.split('.')[-1].lower()
if file_extension == 'csv':
    INPUT_TYPE = 'csv'
elif file_extension in ['xlsx', 'xls']:
    INPUT_TYPE = 'excel'
elif file_extension == 'json':
    INPUT_TYPE = 'json'
else:
    raise ValueError(f"❌ Unsupported file type: {file_extension}. Use CSV, Excel, or JSON.")

print(f"📊 Detected file type: {INPUT_TYPE.upper()}")
print(f"📁 Processing: {uploaded_file_path}\n")

# ============================================================
# STEP 3: DEFINE MODELS
# ============================================================

print("⚙️ Initializing pipeline...")

class InputTypeEnum(str, Enum):
    CSV = "csv"
    EXCEL = "excel"
    JSON = "json"

class Lead(BaseModel):
    lead_id: int
    name: str
    company: str
    title: Optional[str] = None
    email: Optional[str] = None
    website: Optional[str] = None
    industry: Optional[str] = None
    company_size: Optional[str] = None
    location: Optional[str] = None

    @validator('email')
    def validate_email(cls, v):
        if v and '@' not in v:
            return None
        return v

    @validator('website')
    def validate_website(cls, v):
        if v and not v.startswith(('http://', 'https://')):
            return f"https://{v}"
        return v

class EnrichedLead(BaseModel):
    lead_id: int
    name: str
    company: str
    title: Optional[str] = None
    email: Optional[str] = None
    website: Optional[str] = None
    industry: Optional[str] = None
    company_size: Optional[str] = None
    location: Optional[str] = None
    company_summary: str = ""
    score: float = 0.0
    fit_reasons: List[str] = Field(default_factory=list)
    email_generated: Dict[str, str] = Field(default_factory=dict)

class AgentState(BaseModel):
    input_type: Optional[str] = None
    raw_input: Optional[Any] = None
    leads: List[Lead] = Field(default_factory=list)
    enriched_leads: List[EnrichedLead] = Field(default_factory=list)
    current_lead_index: int = 0
    errors: List[str] = Field(default_factory=list)

    class Config:
        arbitrary_types_allowed = True

# ============================================================
# STEP 4: MCP TOOLS
# ============================================================

class MCPTools:
    @staticmethod
    async def web_search(query: str, max_results: int = 3) -> List[Dict[str, str]]:
        try:
            url = f"https://html.duckduckgo.com/html/?q={requests.utils.quote(query)}"
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            results = []
            for result in soup.find_all('a', class_='result__a', limit=max_results):
                title = result.get_text(strip=True)
                link = result.get('href', '')
                if link:
                    results.append({"title": title, "url": link, "snippet": title})

            return results if results else [{"title": "No results", "url": "", "snippet": ""}]
        except Exception as e:
            logger.error(f"Web search failed: {e}")
            return [{"title": "Search failed", "url": "", "snippet": str(e)}]

    @staticmethod
    async def http_get(url: str, max_length: int = 5000) -> str:
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            for script in soup(["script", "style", "nav", "footer", "header"]):
                script.decompose()

            text = soup.get_text(separator=' ', strip=True)
            text = re.sub(r'\s+', ' ', text)[:max_length]
            return text
        except Exception as e:
            logger.error(f"HTTP GET failed for {url}: {e}")
            return f"Failed to fetch: {str(e)}"

    @staticmethod
    def load_csv(file_path: str) -> pd.DataFrame:
        return pd.read_csv(file_path)

    @staticmethod
    def load_excel(file_path: str) -> pd.DataFrame:
        return pd.read_excel(file_path)

    @staticmethod
    def load_json(file_path: str) -> List[Dict]:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return data if isinstance(data, list) else [data]

# ============================================================
# STEP 5: LEAD PIPELINE WITH ELITE PROMPTS
# ============================================================

class LeadPipeline:
    def __init__(self, openai_api_key: str):
        self.client = AsyncOpenAI(api_key=openai_api_key)
        self.tools = MCPTools()
        self.graph = self._build_graph()

    def _build_graph(self) -> StateGraph:
        workflow = StateGraph(AgentState)
        workflow.add_node("input_router", self.input_router_node)
        workflow.add_node("enrich_lead", self.enrich_lead_node)
        workflow.add_node("score_lead", self.score_lead_node)
        workflow.add_node("write_email", self.write_email_node)
        workflow.add_node("output", self.output_node)

        workflow.set_entry_point("input_router")
        workflow.add_edge("input_router", "enrich_lead")
        workflow.add_conditional_edges(
            "enrich_lead",
            self._should_continue,
            {"continue": "enrich_lead", "next": "score_lead"}
        )
        workflow.add_edge("score_lead", "write_email")
        workflow.add_conditional_edges(
            "write_email",
            self._should_continue_email,
            {"continue": "write_email", "next": "output"}
        )
        workflow.add_edge("output", END)
        return workflow.compile()

    def _should_continue(self, state: AgentState) -> Literal["continue", "next"]:
        return "continue" if state.current_lead_index < len(state.leads) else "next"

    def _should_continue_email(self, state: AgentState) -> Literal["continue", "next"]:
        unenriched = [l for l in state.enriched_leads if not l.email_generated]
        return "continue" if unenriched else "next"

    # ========================================
    # AGENT 1: INPUT ROUTER
    # ========================================

    async def input_router_node(self, state: AgentState) -> AgentState:
        logger.info(f"Input Router: Processing {state.input_type} input")
        try:
            if state.input_type == InputTypeEnum.CSV.value:
                df = self.tools.load_csv(state.raw_input)
                state.leads = self._df_to_leads(df)
            elif state.input_type == InputTypeEnum.EXCEL.value:
                df = self.tools.load_excel(state.raw_input)
                state.leads = self._df_to_leads(df)
            elif state.input_type == InputTypeEnum.JSON.value:
                data = self.tools.load_json(state.raw_input)
                state.leads = self._json_to_leads(data)
            logger.info(f"Parsed {len(state.leads)} leads")
        except Exception as e:
            logger.error(f"Input routing failed: {e}")
            state.errors.append(f"Input routing error: {str(e)}")
        return state

    def _df_to_leads(self, df: pd.DataFrame) -> List[Lead]:
        leads = []
        df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]
        column_mapping = {
            'name': ['name', 'contact_name', 'full_name', 'person'],
            'company': ['company', 'organization', 'company_name', 'org'],
            'title': ['title', 'job_title', 'position', 'role'],
            'email': ['email', 'email_address', 'contact_email'],
            'website': ['website', 'url', 'company_url', 'site'],
            'industry': ['industry', 'sector', 'vertical'],
            'company_size': ['company_size', 'size', 'employees', 'employee_count'],
            'location': ['location', 'city', 'country', 'region']
        }

        for idx, row in df.iterrows():
            lead_data = {'lead_id': idx + 1}
            for field, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns and pd.notna(row[col]):
                        lead_data[field] = str(row[col]).strip()
                        break
            try:
                if 'name' in lead_data and 'company' in lead_data:
                    leads.append(Lead(**lead_data))
            except Exception as e:
                logger.warning(f"Skipping invalid row {idx}: {e}")
        return leads

    def _json_to_leads(self, data: List[Dict]) -> List[Lead]:
        leads = []
        for idx, item in enumerate(data):
            try:
                item['lead_id'] = idx + 1
                leads.append(Lead(**item))
            except Exception as e:
                logger.warning(f"Skipping invalid record {idx}: {e}")
        return leads

    # ========================================
    # AGENT 2: COMPANY ENRICHMENT (ELITE PROMPTS)
    # ========================================

    async def enrich_lead_node(self, state: AgentState) -> AgentState:
        if state.current_lead_index >= len(state.leads):
            return state

        lead = state.leads[state.current_lead_index]
        logger.info(f"Enriching lead {lead.lead_id}: {lead.name} at {lead.company}")
        enriched = EnrichedLead(**lead.model_dump())

        try:
            search_results = await self.tools.web_search(f"{lead.company} company")
            website_content = ""
            if lead.website:
                website_content = await self.tools.http_get(lead.website)

            # ⭐ ELITE ENRICHMENT PROMPT ⭐
            system_prompt = """You are an elite B2B research analyst with 15+ years of experience in competitive intelligence and market research. Your expertise includes:

- Deep company analysis across industries (SaaS, FinTech, E-commerce, Enterprise Software)
- Competitive landscape assessment and market positioning
- Business model evaluation and revenue stream analysis
- Technology stack identification and digital maturity assessment
- Growth trajectory analysis and expansion strategy insights

Your analytical approach:
1. Extract key business metrics and market indicators
2. Identify core value propositions and competitive advantages
3. Assess target market segments and customer profiles
4. Evaluate innovation capabilities and product differentiation
5. Synthesize insights into clear, actionable intelligence

Communication style:
- Concise, precise, and data-driven
- Focus on strategic insights over superficial details
- Use business terminology appropriately
- Highlight unique differentiators and market position
- Maintain objectivity and analytical rigor

Constraints:
- Output must be 2-3 sentences maximum
- Include: primary business focus, key products/services, market position/scale
- Avoid marketing fluff or generic descriptions
- Prioritize quantifiable metrics when available"""

            user_prompt = f"""Conduct a comprehensive analysis of this company and synthesize your findings into a strategic summary.

**COMPANY INTELLIGENCE DOSSIER**

Primary Target: {lead.company}

**Data Sources:**

1. WEBSITE CONTENT ANALYSIS:
{website_content[:1000] if website_content else "No website content available"}

2. WEB RESEARCH FINDINGS:
{json.dumps(search_results[:2], indent=2) if search_results else "No search results available"}

3. AVAILABLE CONTEXT:
- Contact: {lead.name}
- Role: {lead.title or 'Unknown'}
- Industry Indicator: {lead.industry or 'To be determined'}
- Location: {lead.location or 'Unknown'}

**ANALYSIS REQUIREMENTS:**

Your summary must address:
1. **Core Business**: What does this company actually do? (specific products/services)
2. **Market Position**: Scale, reach, or notable achievements (use quantifiable metrics if available)
3. **Strategic Context**: Industry vertical, business model, or unique differentiators

**OUTPUT FORMAT:**

Deliver exactly 2-3 sentences that a C-level executive could use to understand this company in 30 seconds. Focus on:
- Concrete business activities (not vague descriptions)
- Market indicators (size, growth, position)
- Strategic relevance (why this company matters in their space)

Begin your analysis."""

            response = await self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=250,
                temperature=0.3
            )
            enriched.company_summary = response.choices[0].message.content.strip()

            # Infer industry if missing
            if not enriched.industry:
                enriched.industry = await self._infer_industry(lead.company, enriched.company_summary)

            logger.info(f"✓ Enriched lead {lead.lead_id}")
        except Exception as e:
            logger.error(f"Enrichment failed: {e}")
            enriched.company_summary = "Unable to enrich"

        state.enriched_leads.append(enriched)
        state.current_lead_index += 1
        return state

    async def _infer_industry(self, company: str, summary: str) -> str:
        try:
            # ⭐ ELITE INDUSTRY CLASSIFICATION PROMPT ⭐
            system_prompt = """You are a senior industry classification specialist with expertise in:
- GICS (Global Industry Classification Standard)
- NAICS (North American Industry Classification System)
- Market taxonomy and sector analysis
- Emerging technology categorization
- Cross-industry convergence patterns

Your role is to classify companies into precise industry categories based on their business model, products, and market positioning.

Classification principles:
1. Use standard industry terminology (avoid creating new categories)
2. Be specific (e.g., "SaaS Collaboration Tools" not just "Software")
3. Capture primary business focus (not adjacent activities)
4. Consider modern categories (AI/ML, FinTech, HealthTech, etc.)
5. Use singular form (e.g., "Financial Technology" not "Financial Technologies")

Common industries to choose from:
- Financial Technology (FinTech)
- Enterprise Software / SaaS
- E-commerce Platform
- Cloud Infrastructure
- Cybersecurity
- Data Analytics & Business Intelligence
- Marketing Technology (MarTech)
- Healthcare Technology (HealthTech)
- Productivity Software
- Developer Tools
- Design & Creative Software
- Communication & Collaboration
- Payment Processing
- Customer Relationship Management (CRM)
- Human Resources Technology (HRTech)

Respond with ONLY the industry name. No explanation, no additional text."""

            user_prompt = f"""Classify this company into a precise industry category.

**COMPANY ANALYSIS:**

Company Name: {company}

Business Summary: {summary[:300]}

**CLASSIFICATION TASK:**

Based on the company's core business model and primary revenue streams, determine the single most accurate industry classification.

Output format: Return ONLY the industry name (2-4 words maximum).

Industry classification:"""

            response = await self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=20,
                temperature=0.1
            )
            return response.choices[0].message.content.strip()
        except:
            return "Unknown"

    # ========================================
    # AGENT 3: LEAD SCORING (ELITE PROMPTS)
    # ========================================

    async def score_lead_node(self, state: AgentState) -> AgentState:
        logger.info(f"Scoring {len(state.enriched_leads)} leads")
        for lead in state.enriched_leads:
            try:
                # ⭐ ELITE SCORING PROMPT ⭐
                system_prompt = """You are an elite B2B sales qualification expert with deep expertise in:

**QUALIFICATION FRAMEWORKS:**
- BANT (Budget, Authority, Need, Timeline)
- MEDDIC (Metrics, Economic Buyer, Decision Criteria, Decision Process, Identify Pain, Champion)
- CHAMP (Challenges, Authority, Money, Prioritization)
- Ideal Customer Profile (ICP) matching

**EVALUATION DOMAINS:**
1. **Organizational Fit**: Company size, industry, market position, growth stage
2. **Decision-Making Authority**: Title seniority, functional role, purchasing power
3. **Engagement Potential**: Company maturity, technology adoption, expansion signals
4. **Strategic Alignment**: Business challenges, pain points, solution fit
5. **Accessibility**: Contact information quality, engagement channels

**SCORING METHODOLOGY:**

10/10 - IDEAL PROSPECT (Top 5%)
- C-level/VP at high-growth company ($50M+ revenue or Series B+)
- Direct decision-making authority with confirmed budget
- Company actively expanding/hiring (strong growth signals)
- Perfect ICP match with urgent, identifiable needs

9/10 - EXCELLENT FIT (Top 10%)
- Senior leadership (Director+) at established company
- Strategic role with influence over significant budgets
- Company showing strong growth indicators
- High probability of conversion

8/10 - STRONG OPPORTUNITY (Top 20%)
- Management level (Manager+) at stable, growing company
- Role involves relevant decision-making or strong influence
- Company has clear use case
- Moderate-to-high conversion probability

7/10 - QUALIFIED LEAD (Top 35%)
- Mid-level professional at recognizable company
- Some decision influence
- Requires nurturing but has potential

6/10 - DEVELOPING OPPORTUNITY (Top 50%)
- Individual contributor at established company
- Limited direct authority but potential champion
- Long-term potential but requires patience

5/10 or below - Lower priority

**OUTPUT REQUIREMENTS:**
- Provide single numeric score (1-10, can use decimals like 8.5)
- List 3-5 specific, data-driven reasons
- Base reasoning on observable facts from the data

Return ONLY valid JSON, no other text."""

                user_prompt = f"""Conduct a comprehensive B2B lead qualification analysis and assign a precision score.

**LEAD INTELLIGENCE PROFILE**

**CONTACT INFORMATION:**
Full Name: {lead.name}
Job Title: {lead.title or 'Not provided'}
Email: {lead.email or 'Not available'}

**ORGANIZATION PROFILE:**
Company: {lead.company}
Industry: {lead.industry or 'Not specified'}
Company Size: {lead.company_size or 'Unknown'}
Location: {lead.location or 'Unknown'}
Website: {lead.website or 'Not available'}

**BUSINESS INTELLIGENCE:**
{lead.company_summary if lead.company_summary else "No company summary available"}

**QUALIFICATION ANALYSIS REQUIRED:**

Evaluate this lead across these dimensions:

1. **AUTHORITY & INFLUENCE** (Weight: 30%)
   - Job title seniority
   - Decision-making scope and budget authority
   - Role relevance to typical buying committee

2. **ORGANIZATIONAL FIT** (Weight: 25%)
   - Company size and growth trajectory
   - Industry alignment with ICP
   - Market position and maturity stage

3. **ENGAGEMENT POTENTIAL** (Weight: 25%)
   - Contact information completeness
   - Company visibility and accessibility
   - Technology adoption indicators

4. **STRATEGIC VALUE** (Weight: 20%)
   - Business challenges alignment
   - Potential deal size
   - Long-term account potential

**SCORING INSTRUCTIONS:**

Assign a score from 1.0 to 10.0 based on the weighted methodology above. Use decimal precision (e.g., 8.7, 6.3).

**OUTPUT FORMAT (JSON only):**

{{
  "score": <float between 1.0 and 10.0>,
  "reasons": [
    "<specific reason 1 with data point>",
    "<specific reason 2 with data point>",
    "<specific reason 3 with data point>"
  ]
}}

Begin qualification analysis:"""

                response = await self.client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=400,
                    temperature=0.5
                )
                result = json.loads(response.choices[0].message.content.strip())
                lead.score = float(result.get('score', 5.0))
                lead.fit_reasons = result.get('reasons', ['Qualified lead'])
                logger.info(f"✓ Scored lead {lead.lead_id}: {lead.score}/10")
            except Exception as e:
                logger.error(f"Scoring failed: {e}")
                lead.score = 5.0
                lead.fit_reasons = ["Unable to score"]
        return state

    # ========================================
    # AGENT 4: EMAIL WRITER (ELITE PROMPTS)
    # ========================================

    async def write_email_node(self, state: AgentState) -> AgentState:
        logger.info(f"Writing emails for {len(state.enriched_leads)} leads")
        for lead in state.enriched_leads:
            if lead.email_generated:
                continue
            try:
                # ⭐ ELITE EMAIL COPYWRITING PROMPT ⭐
                system_prompt = """You are an elite B2B email copywriter and sales communication strategist with expertise in:

**COPYWRITING EXCELLENCE:**
- High-converting cold email frameworks (AIDA, PAS, BAB)
- Personalization at scale with authentic voice
- Value-based selling and consultative approach
- Executive-level communication standards
- Multi-industry B2B messaging

**STRATEGIC PRINCIPLES:**

1. **PERSONALIZATION DEPTH:**
   - Reference specific company context
   - Acknowledge recipient's role and likely challenges
   - Demonstrate research and genuine understanding

2. **VALUE PROPOSITION:**
   - Lead with recipient benefit, not sender agenda
   - Connect solution to specific business outcomes
   - Position as strategic partner, not vendor

3. **CREDIBILITY SIGNALS:**
   - Subtle social proof (similar companies, results)
   - Industry knowledge and thought leadership
   - Problem awareness that demonstrates expertise

4. **ENGAGEMENT PSYCHOLOGY:**
   - Open strong (compelling subject + first sentence)
   - Use conversational but professional language
   - Create curiosity gaps that encourage response
   - One clear, low-friction call-to-action

**STRUCTURAL FRAMEWORK:**

SUBJECT LINE (5-8 words):
- Specific to their company/situation
- Intriguing but not clickbait
- Avoid spam triggers

OPENING (1-2 sentences):
- Personalized hook referencing research
- Establish credibility and relevance

VALUE PROPOSITION (2-3 sentences):
- Core message focused on THEIR benefit
- Specific business outcome addressed

CALL-TO-ACTION (1 sentence):
- Low commitment ask (15-min call, quick question)
- Specific and actionable

**TONE CALIBRATION:**

Executive/C-Level: Direct, concise, strategic focus
Director/VP: Balance of strategic and tactical
Manager/IC: Practical, solution-oriented

Return ONLY valid JSON."""

                user_prompt = f"""Craft a high-converting, personalized B2B outreach email for this qualified lead.

**LEAD QUALIFICATION PROFILE**

**TARGET CONTACT:**
Name: {lead.name}
Title: {lead.title or 'Professional'}
Role Context: {'Senior decision-maker' if any(x in str(lead.title).lower() for x in ['vp', 'director', 'chief', 'head']) else 'Professional'}

**ORGANIZATION INTELLIGENCE:**
Company: {lead.company}
Industry: {lead.industry or 'Unknown'}
Scale: {lead.company_size or 'Unknown'} employees
Location: {lead.location or 'Unknown'}

**BUSINESS CONTEXT:**
{lead.company_summary if lead.company_summary else f"{lead.company} operates in the {lead.industry or 'technology'} space."}

**QUALIFICATION ASSESSMENT:**
Fit Score: {lead.score}/10.0
Top Qualification Factors:
{chr(10).join(f'  • {reason}' for reason in lead.fit_reasons[:3])}

**EMAIL COMPOSITION REQUIREMENTS:**

1. **RESEARCH-BASED PERSONALIZATION:**
   - Reference specific company context
   - Acknowledge their role and likely priorities
   - Make it clear this is NOT a mass email

2. **VALUE-FIRST APPROACH:**
   - Open with insight about THEIR business
   - Focus on business outcomes relevant to their role
   - Position as strategic partner

3. **STRATEGIC CALL-TO-ACTION:**
   - Appropriate for their seniority level
   - Low-commitment ask (15 minutes, quick chat)
   - Specific next step

**CRITICAL CONSTRAINTS:**
✓ Total length: 125-175 words (body only)
✓ Subject line: 5-8 words, company-specific
✓ Paragraphs: 2-3 lines maximum each
✓ Format: Conversational prose
✓ End with simple "Best," or "Best regards,"

**OUTPUT FORMAT (JSON only):**

{{
  "subject": "<5-8 word subject line>",
  "body": "<125-175 word email body>"
}}

Now craft the email:"""

                response = await self.client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=500,
                    temperature=0.7
                )
                result = json.loads(response.choices[0].message.content.strip())
                lead.email_generated = {
                    "subject": result.get('subject', 'Partnership Opportunity'),
                    "body": result.get('body', 'Email generation failed')
                }
                logger.info(f"✓ Generated email for lead {lead.lead_id}")
            except Exception as e:
                logger.error(f"Email generation failed: {e}")
                lead.email_generated = {"subject": "Connection", "body": "Failed to generate"}
        return state

    async def output_node(self, state: AgentState) -> AgentState:
        logger.info(f"Pipeline complete: {len(state.enriched_leads)} leads processed")
        return state

print("✅ Pipeline initialized\n")

# ============================================================
# STEP 6: RUN PIPELINE
# ============================================================

print("="*70)
print("🚀 STARTING LEAD GENERATION PIPELINE")
print("="*70)
print()

async def run_full_pipeline():
    pipeline = LeadPipeline(OPENAI_API_KEY)
    initial_state = AgentState(input_type=INPUT_TYPE, raw_input=uploaded_file_path)
    final_state = await pipeline.graph.ainvoke(initial_state)

    output = []
    enriched_leads = final_state.get('enriched_leads', []) if isinstance(final_state, dict) else final_state.enriched_leads

    for lead in enriched_leads:
        lead_dict = lead.model_dump() if hasattr(lead, 'model_dump') else (lead.dict() if hasattr(lead, 'dict') else lead)
        output.append({
            "lead_id": lead_dict["lead_id"],
            "name": lead_dict["name"],
            "company": lead_dict["company"],
            "title": lead_dict["title"],
            "email": lead_dict["email"],
            "website": lead_dict["website"],
            "industry": lead_dict.get("industry"),
            "company_size": lead_dict.get("company_size"),
            "location": lead_dict.get("location"),
            "company_summary": lead_dict["company_summary"],
            "score": round(lead_dict["score"], 1),
            "fit_reasons": lead_dict["fit_reasons"],
            "email_generated": lead_dict["email_generated"]
        })
    return output

results = asyncio.run(run_full_pipeline())

print()
print("="*70)
print("✅ PIPELINE COMPLETE")
print("="*70)
print()

# ============================================================
# STEP 7: DISPLAY & SAVE RESULTS
# ============================================================

print(f"📊 Processed {len(results)} leads\n")

if results:
    print("📋 SAMPLE ENRICHED LEAD:")
    print("="*70)
    print(json.dumps(results[0], indent=2))
    print()

print("\n📈 LEAD SUMMARY:")
print("="*70)
summary_df = pd.DataFrame([{
    'Name': r['name'],
    'Company': r['company'],
    'Score': r['score'],
    'Industry': r.get('industry') or 'Unknown',
    'Email Subject': r['email_generated']['subject'][:50] + '...'
} for r in results])
print(summary_df.to_string(index=False))
print()

# Save outputs
print("💾 Saving outputs...")
with open('enriched_leads.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✅ Saved: enriched_leads.json")

output_df = pd.DataFrame(results)
output_df['email_subject'] = output_df['email_generated'].apply(lambda x: x['subject'])
output_df['email_body'] = output_df['email_generated'].apply(lambda x: x['body'])
output_df['fit_reasons_text'] = output_df['fit_reasons'].apply(lambda x: '; '.join(x))
output_df = output_df.drop(['email_generated', 'fit_reasons'], axis=1)
output_df.to_csv('enriched_leads.csv', index=False)
print("✅ Saved: enriched_leads.csv")

# Statistics
print()
print("📊 PIPELINE STATISTICS:")
print("="*70)
print(f"Environment: {ENV_TYPE}")
print(f"Input File: {uploaded_file_path}")
print(f"File Type: {INPUT_TYPE.upper()}")
print(f"Total Leads Processed: {len(results)}")
print(f"Average Score: {sum(r['score'] for r in results) / len(results):.1f}/10")
print(f"High-Value Leads (>8.0): {len([r for r in results if r['score'] >= 8.0])}")
print(f"Emails Generated: {len([r for r in results if r['email_generated']])}")
print()

print("🎉 Done! Check enriched_leads.json and enriched_leads.csv")
print()
print("💡 TIP: Access results:")
print("   - results[0] → First lead")
print("   - results[0]['email_generated'] → Email")
print("   - [r for r in results if r['score'] >= 8] → High-scoring leads")

🔑 API Key loaded from Colab Secrets
✅ API Key validated (Environment: Colab)

📂 FILE UPLOAD
🔵 Running in Google Colab
Please upload your file (CSV, Excel, or JSON):



Saving sample_leads.csv to sample_leads.csv
✅ Uploaded: sample_leads.csv

📊 Detected file type: CSV
📁 Processing: sample_leads.csv

⚙️ Initializing pipeline...
✅ Pipeline initialized

🚀 STARTING LEAD GENERATION PIPELINE



/tmp/ipython-input-631373340.py:134: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator('email')
/tmp/ipython-input-631373340.py:140: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator('website')
/tmp/ipython-input-631373340.py:161: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at ht


✅ PIPELINE COMPLETE

📊 Processed 5 leads

📋 SAMPLE ENRICHED LEAD:
{
  "lead_id": 1,
  "name": "Sarah Chen",
  "company": "Stripe",
  "title": "VP of Product",
  "email": "sarah.chen@stripe.com",
  "website": "https://stripe.com",
  "industry": "Financial Technology",
  "company_size": "5000+",
  "location": "San Francisco",
  "company_summary": "Stripe is a leading financial technology company that provides a comprehensive suite of payment processing solutions, enabling businesses to accept online and in-person payments, manage subscriptions, and integrate financial services into their platforms. With a reported net payment volume of over \u20ac3.5 million in a single day and a customer growth rate of 32.1%, Stripe is positioned as a dominant player in the global payments landscape, serving millions of businesses worldwide. Its unique value proposition lies in its flexible infrastructure that supports diverse business models, making it a critical partner for companies looking to enhan